In [1]:
# Run this cell first. It installs the packages this notebook uses.
%pip install -q pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


# 01. Presidential elections since 1976

## The ideas first, in plain words

**Popular vote.** Every vote cast for president, added up across the whole
country. The candidate with the most popular votes wins the popular vote.

**Electoral College.** The president is not chosen directly by the national
popular vote. Each state gets a number of electors, roughly based on its
population. In almost every state, the candidate who wins the state wins all
of that state's electors. There are 538 electors in total. A candidate needs
270 to win.

**Swing state.** A state where either party can plausibly win, so the result
is close and both campaigns spend a lot of time there. A safe state is one
where the winner is almost never in doubt.

**Turnout.** The share of eligible voters who actually vote. This project
uses voting eligible population turnout, which counts only people who are
legally allowed to vote, not every adult resident.

**Wrong winner election.** An election where the candidate who won the most
popular votes lost the Electoral College, and therefore lost the presidency.

With those ideas in hand, here is what the data shows.

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt

PROC = os.path.join('..', 'data', 'processed')
CHART_DIR = os.path.join('..', 'charts')
os.makedirs(CHART_DIR, exist_ok=True)
plt.rcParams['figure.dpi'] = 120
DEM = '#1f77b4'
REP = '#d62728'
GREY = '#7f7f7f'

def savefig(name):
    path = os.path.join(CHART_DIR, name)
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    print('saved', path)

In [3]:
nat = pd.read_csv(os.path.join(PROC, 'presidential_national.csv'))
state = pd.read_csv(os.path.join(PROC, 'presidential_state.csv'))
swing = pd.read_csv(os.path.join(PROC, 'swing_states.csv'))
print(nat[['year', 'dem_nominee', 'rep_nominee', 'dem_share_pct', 'rep_share_pct',
           'pop_margin_pp', 'dem_ev', 'rep_ev', 'vep_turnout_pct']].to_string(index=False))

 year         dem_nominee       rep_nominee  dem_share_pct  rep_share_pct  pop_margin_pp  dem_ev  rep_ev  vep_turnout_pct
 1976       CARTER, JIMMY      FORD, GERALD          49.85          47.64           2.22     297     240              NaN
 1980       CARTER, JIMMY    REAGAN, RONALD          41.02          50.46           9.44      49     489            58.49
 1984     MONDALE, WALTER    REAGAN, RONALD          40.42          58.46          18.04      13     525            55.25
 1988    DUKAKIS, MICHAEL BUSH, GEORGE H.W.          45.55          53.11           7.56     111     426            52.76
 1992       CLINTON, BILL BUSH, GEORGE H.W.          42.88          37.09           5.79     370     168            58.22
 1996       CLINTON, BILL      DOLE, ROBERT          49.07          40.46           8.60     379     159            51.73
 2000            GORE, AL   BUSH, GEORGE W.          48.14          47.65           0.49     266     271            54.24
 2004         KERRY, JOH

## Popular vote share over time

Each line is one party's share of the national popular vote. The two lines
almost always add up to close to 100 percent, with the rest going to
third party candidates. Notice 1992: the Democrat won with only about 43
percent, because Ross Perot took about 19 percent as an independent.

In [4]:
plt.figure(figsize=(10, 5))
plt.plot(nat['year'], nat['dem_share_pct'], marker='o', color=DEM, label='Democrat')
plt.plot(nat['year'], nat['rep_share_pct'], marker='o', color=REP, label='Republican')
plt.axhline(50, color=GREY, linestyle='--', linewidth=1)
plt.ylabel('Share of popular vote (%)')
plt.xlabel('Election year')
plt.title('Popular vote share by party, 1976 to 2024')
plt.legend()
plt.grid(alpha=0.3)
savefig('c01_popular_share.png')

saved ../charts/c01_popular_share.png


## The popular vote margin, and the two wrong winners

This chart shows how many percentage points separated the top two
candidates. Small bars mean close elections. The two red outlines mark the
wrong winner elections, where the popular vote winner lost the presidency.

In [5]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = [REP if w else DEM for w in nat['wrong_winner']]
ax.bar(nat['year'].astype(str), nat['pop_margin_pp'], color=DEM)
for i, w in enumerate(nat['wrong_winner']):
    if w:
        ax.patches[i].set_facecolor('#f4b400')
        ax.patches[i].set_edgecolor('black')
ax.set_ylabel('Popular vote margin (percentage points)')
ax.set_xlabel('Election year')
ax.set_title('How close was it? Popular vote margin, 1976 to 2024 (gold = wrong winner)')
ax.grid(axis='y', alpha=0.3)
savefig('c01_popular_margin.png')

saved ../charts/c01_popular_margin.png


In [6]:
for _, r in nat[nat['wrong_winner']].iterrows():
    gap = abs(int(r['dem_votes']) - int(r['rep_votes']))
    print(int(r['year']), ': popular vote gap was', f'{gap:,}', 'votes, but the')
    print('   Electoral College went', int(r['dem_ev']), 'to', int(r['rep_ev']),
          'for the', 'Democrat' if r['ev_winner'] == 'D' else 'Republican', 'candidate.')
print()
print('2000 detail: Al Gore won the popular vote by 519,208 votes, but George W. Bush')
print('won the Electoral College 271 to 266. One District of Columbia elector abstained,')
print('so only 537 electoral votes were cast.')
print('2016 detail: Hillary Clinton won the popular vote by 2,984,618 votes, but Donald')
print('Trump won the Electoral College 304 to 227. Seven electors voted for someone else,')
print('so the two main candidates together received only 531 electoral votes.')

2000 : popular vote gap was 519,208 votes, but the
   Electoral College went 266 to 271 for the Republican candidate.
2016 : popular vote gap was 2,984,618 votes, but the
   Electoral College went 227 to 304 for the Republican candidate.

2000 detail: Al Gore won the popular vote by 519,208 votes, but George W. Bush
won the Electoral College 271 to 266. One District of Columbia elector abstained,
so only 537 electoral votes were cast.
2016 detail: Hillary Clinton won the popular vote by 2,984,618 votes, but Donald
Trump won the Electoral College 304 to 227. Seven electors voted for someone else,
so the two main candidates together received only 531 electoral votes.


## Electoral College votes vs popular votes

Each dot is one election. The horizontal axis is the popular vote margin,
the vertical axis is the Electoral College margin. Dots in the top right and
bottom left are elections where both agreed. The two dots that break the
pattern are 2000 and 2016: the popular vote went one way, the Electoral
College went the other.

In [7]:
nat['ev_margin'] = (nat['dem_ev'] - nat['rep_ev'])
nat['pop_signed'] = nat.apply(lambda r: r['pop_margin_pp'] if r['pop_winner'] == 'D' else -r['pop_margin_pp'], axis=1)
plt.figure(figsize=(8, 6))
plt.scatter(nat['pop_signed'], nat['ev_margin'], s=80, color=DEM)
for _, r in nat.iterrows():
    plt.annotate(str(int(r['year'])), (r['pop_signed'], r['ev_margin']), fontsize=9,
                 xytext=(4, 4), textcoords='offset points')
plt.axhline(0, color=GREY, linewidth=1)
plt.axvline(0, color=GREY, linewidth=1)
plt.xlabel('Popular vote margin (points, positive = Democrat ahead)')
plt.ylabel('Electoral College margin (Democrat minus Republican)')
plt.title('Popular vote vs Electoral College, 1976 to 2024')
plt.grid(alpha=0.3)
savefig('c01_ev_vs_pop.png')

saved ../charts/c01_ev_vs_pop.png


## Turnout: who actually votes

Turnout in presidential elections has risen since the 1990s. The lowest
point in this period was 1996, at 51.73 percent of eligible voters. The
highest was 2020, at 65.28 percent. The 2024 figure, 64.30 percent, is a
provisional estimate from the University of Florida Election Lab, dated
March 2, 2026, and may be revised.

In [8]:
t = nat.dropna(subset=['vep_turnout_pct'])
plt.figure(figsize=(10, 5))
plt.plot(t['year'], t['vep_turnout_pct'], marker='o', color='#2ca02c')
plt.ylabel('Voting eligible population turnout (%)')
plt.xlabel('Election year')
plt.title('Presidential election turnout, 1980 to 2024')
plt.grid(alpha=0.3)
savefig('c01_turnout.png')
print('Lowest turnout in this window:', t.loc[t['vep_turnout_pct'].idxmin(), 'year'],
      t['vep_turnout_pct'].min(), 'percent.')
print('Highest turnout in this window:', t.loc[t['vep_turnout_pct'].idxmax(), 'year'],
      t['vep_turnout_pct'].max(), 'percent.')

saved ../charts/c01_turnout.png
Lowest turnout in this window: 1996 51.73 percent.
Highest turnout in this window: 2020 65.28 percent.


## Swing states: how many states are actually close

A state counts as close here if the winner's margin was under 5 percentage
points. The number of close states bounces around a lot. It was 19 in 1976
and 18 in 1992, but only 3 in the 1984 landslide and 4 in 2012. In 2020 and
2024 it was 8 each. There is no steady decline in competitiveness in this
measure. The map changes, but close states keep appearing.

In [9]:
plt.figure(figsize=(10, 5))
plt.bar(swing['year'].astype(str), swing['states_under_5pp'], color='#9467bd')
plt.ylabel('States decided by under 5 points')
plt.xlabel('Election year')
plt.title('Close states per presidential election, 1976 to 2024')
plt.grid(axis='y', alpha=0.3)
savefig('c01_swing_states.png')
print(swing.to_string(index=False))

saved ../charts/c01_swing_states.png
 year  states_under_5pp
 1976                19
 1980                16
 1984                 3
 1988                11
 1992                18
 1996                11
 2000                12
 2004                11
 2008                 6
 2012                 4
 2016                11
 2020                 8
 2024                 8


## What to take into the next chapter

Presidential elections are usually close, sometimes extremely close, and
twice since 1976 the popular vote winner lost. Turnout is higher than it
was in the 1990s, and a handful of close states decide most elections.
Midterms, the subject of the next chapter, work differently: the president
is not on the ballot, turnout is lower, and the president's party usually
loses seats.